# HealthBot

## Imports & env Configuration

In [1]:
# Standard Library
import os
import re
import time
from typing import TypedDict, Any

# Environment Variables
from dotenv import load_dotenv

# LangChain / LangGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END

# Tavily
from tavily import TavilyClient

# Local Topic Validation
import numpy as np
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer


print("All imports loaded successfully!")

All imports loaded successfully!


## Environment Key Configuration

In [2]:
load_dotenv("config.env")

print("Environment variables loaded successfully!")

OPENAI_API_KEY = os.getenv("PERSONAL_OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("PERSONAL_GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("PERSONAL_TAVILY_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY is missing. Check your config.env file."
    )
else:
    print("OpenAI key loaded:", bool(OPENAI_API_KEY))
    
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is missing. Check your config.env file."
    )
else:
    print("Gemini key loaded:", bool(GEMINI_API_KEY))

if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY is missing. Check your config.env file."
    )
else:
    print("Tavily key loaded:", bool(TAVILY_API_KEY))

print("API credentials are configured correctly.")

Environment variables loaded successfully!
OpenAI key loaded: True
Gemini key loaded: True
Tavily key loaded: True
API credentials are configured correctly.


## External API Test

In [ ]:
# OpenAI API Test
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)
print("OpenAI model configured.")
try:
    response = llm.invoke(
        "Respond with exactly: OpenAI API is working."
    )

    print(response.content)

except Exception as e:
    print("OpenAI API test failed.")
    print(f"Error: {e}")

# Gemini API Test
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)
print("-----------------------\n")
print("Gemini model configured successfully.")
try:
    response = llm.invoke(
        "Respond with exactly: Gemini API is working."
    )

    print(response.content)

except Exception as e:
    print("Gemini API test failed.")
    print(f"Error: {e}")

# Tavily API Test
tavily_tool = TavilySearch(
    max_results=1,
    tavily_api_key=TAVILY_API_KEY
)
print("-----------------------\n")
print("Tavily tool configured.")
try:
    tavily_response = tavily_tool.invoke(
        {
            "query": "diabetes symptoms reputable medical sources"
        }
    )
    # Check overall response
    if not isinstance(tavily_response, dict):
        print("Tavily returned an unexpected response type.")
        print("Type:", type(tavily_response))
    else:
        print("Tavily API is working.")
        print("Response type:", type(tavily_response))
        # Extract actual search results
        search_results = tavily_response.get("results", [])
        if not search_results:
            print("Tavily returned no search results.")
        else:
            print(f"Search results returned: {len(search_results)}")
            for i, result in enumerate(search_results, start=1):
                print(f"--- Result {i} ---")
                print("Type:", type(result))
                if isinstance(result, dict):
                    print("Title:", result.get("title", "N/A"))
                    print("URL:", result.get("url", "N/A"))
                    content = result.get("content", "")
                    print(
                        "Content:",
                        content[:50]+"..." if content else "N/A"
                    )
                    print(
                        "Score:",
                        result.get("score", "N/A")
                    )
                else:
                    print("Result:", result)
except Exception as e:
    print("Tavily API test failed.")
    print(f"Error: {type(e).__name__}: {e}")

## AI Components

This section configures the Gemini language model and Tavily search tool that will be used by the HealthBot LangGraph workflow.

In [3]:
# Gemini Model Configuration
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

# Tavily Model Configuration
tavily_tool = TavilySearch(
    max_results=5,
    tavily_api_key=TAVILY_API_KEY
)

## Helper Functions

In [4]:
# Check whether a value is a non-empty string.
def is_valid_text(value: Any) -> bool:
    return isinstance(value, str) and bool(value.strip())

# Helper function to search for medical information using Tavily.
def search_medical_information(topic: str) -> list[dict[str, Any]]:

    if not is_valid_text(topic):
        raise ValueError("Health topic cannot be empty.")

    def perform_search():
        response = tavily_tool.invoke(
            {
                "query": f"{topic} medical information reputable sources"
            }
        )

        if not isinstance(response, dict):
            raise ValueError(
                "Tavily returned an unexpected response format."
            )
        results = response.get("results", [])
        if not isinstance(results, list):
            raise ValueError(
                "Tavily results are in an unexpected format."
            )

        valid_results = []

        for result in results:
            if not isinstance(result, dict):
                continue

            title = result.get("title", "")
            url = result.get("url", "")
            content = result.get("content", "")

            if not content:
                continue

            valid_results.append(
                {
                    "title": title,
                    "url": url,
                    "content": content,
                    "score": result.get("score", 0)
                }
            )

        if not valid_results:
            raise ValueError(
                "Tavily returned no usable medical information."
            )
        return valid_results

    return retry_operation(perform_search)

# Helper function to retry operations
def retry_operation(operation, max_attempts=3, delay=2):
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            return operation()

        except Exception as e:
            last_error = e

            print(
                f"Attempt {attempt}/{max_attempts} failed: {e}"
            )

            if attempt < max_attempts:
                time.sleep(delay)

    raise RuntimeError(
        f"Operation failed after {max_attempts} attempts: {last_error}"
    )

# Helper function to generate a response using the Gemini LLM. It takes a prompt as input and returns the generated content from the model.
def generate_with_gemini(prompt: str):

    if not prompt or not prompt.strip():
        raise ValueError("Prompt cannot be empty.")

    def generate():
        response = gemini_llm.invoke(prompt)

        # Case 1: LangChain response has .content
        content = getattr(response, "content", None)
        if content is None:
            raise ValueError(
                "Gemini returned no content."
            )

        # Case 2: Gemini content is already a string
        if isinstance(content, str):

            text = content.strip()

        # Case 3: Gemini content is a list
        elif isinstance(content, list):
            text_parts = []
            for item in content:
                if isinstance(item, str):
                    text_parts.append(item)
                elif isinstance(item, dict):
                    if item.get("type") == "text":
                        text = item.get("text", "")
                        if text:
                            text_parts.append(text)
            text = "\n".join(text_parts).strip()
        else:
            text = str(content).strip()

        # Validate final response
        if not text:
            raise ValueError(
                "Gemini returned an empty response."
            )
        return text
    
    return retry_operation(generate)

# Helper function to format prompts for the LLMs. Instead of manually formatting strings each time, we can use this function to format prompts with dynamic content.
def format_prompt(template: str, **kwargs) -> str:
    return template.format(**kwargs)

# Helper function to validate that required fields are not empty. It raises a ValueError if any field is empty or None.
def validate_prompt_data(**kwargs):
    for field, value in kwargs.items():
        if value is None or not str(value).strip():
            raise ValueError(f"{field} cannot be empty.")


# Extract a valid grade from Gemini's grading response.
VALID_GRADES = {"A", "B", "C", "D"}
def extract_grade(feedback: str) -> str:
    if not is_valid_text(feedback):
        return ""

    match = re.search(
        r"\bGrade\s*[:\-]?\s*([ABCD])\b",
        feedback,
        re.IGNORECASE
    )

    if match:
        grade = match.group(1).upper()

        if grade in VALID_GRADES:
            return grade

    return ""

## LangGraph State Definition

HealthBot uses a shared state object to store information that is passed between workflow nodes.

The state contains the user's health topic, retrieved medical information, generated summary, quiz question, user answer, grading result, session decisions, and error information.

In [5]:
class HealthBotState(TypedDict, total=False):
    topic: str
    search_results: list[dict[str, Any]]
    summary: str
    quiz_question: str
    user_answer: str
    grade: str
    feedback: str
    ready_for_quiz: bool
    continue_session: bool
    error: str

In [6]:
# General Error Helper
def create_error_state(message: str) -> HealthBotState:
    return {
        "error": message
    }

In [ ]:
# Initiale State
initial_state: HealthBotState = {
    "topic": "",
    "search_results": [],
    "summary": "",
    "quiz_question": "",
    "user_answer": "",
    "grade": "",
    "feedback": "",
    "ready_for_quiz": False,
    "continue_session": False,
    "error": ""
}
print("Initial State:", initial_state)

In [ ]:
# Testing the state modification 
state: HealthBotState = {
    "topic": "Diabetes",
    "search_results": [{"title": "Diabetes Source"}],
    "summary": "Diabetes summary...",
    "quiz_question": "What is diabetes?",
    "user_answer": "A condition affecting blood glucose.",
    "grade": "A",
    "feedback": "Correct answer.",
    "ready_for_quiz": True,
    "continue_session": True,
    "error": ""
}

print("Topic:", state["topic"])
print("Summary:", state["summary"])
print("Quiz:", state["quiz_question"])
print("Answer:", state["user_answer"])
print("Grade:", state["grade"])
print("Feedback:", state["feedback"])

In [31]:
# State Reset Function, to reset the state to its initial values
def reset_state() -> HealthBotState:
    return {
        "topic": "",
        "search_results": [],
        "summary": "",
        "quiz_question": "",
        "user_answer": "",
        "grade": "",
        "feedback": "",
        "ready_for_quiz": False,
        "continue_session": False,
        "error": ""
    }
# Testing
state = reset_state()
print("Reset State:", state)

Reset State: {'topic': '', 'search_results': [], 'summary': '', 'quiz_question': '', 'user_answer': '', 'grade': '', 'feedback': '', 'ready_for_quiz': False, 'continue_session': False, 'error': ''}


## Prompts

HealthBot uses four dedicated prompts for:

1. Medical information summarization
2. Comprehension quiz generation
3. Patient answer grading
4. Topic classification 

Each prompt has a clearly defined source of information to reduce unsupported or hallucinated content.

In [7]:
# Summarization Prompt
SUMMARY_PROMPT = """
You are HealthBot, an AI-powered patient education assistant.
Your task is to create a clear and patient-friendly explanation about the
health topic using ONLY the medical information provided in the search results.

HEALTH TOPIC: 
{topic}

MEDICAL SEARCH RESULTS:
{search_results}

Instructions:
1. Use ONLY the information provided in the medical search results.
2. Do NOT use outside knowledge.
3. Do NOT invent or assume medical facts.
4. Write exactly 3 to 4 clear paragraphs.
5. Use simple, patient-friendly language.
6. Explain important information that is relevant to understanding the topic.
7. Do not diagnose the patient.
8. Do not provide personalized medical advice.
9. Do not prescribe medication or recommend changing prescribed treatment.
10. If the search results do not contain enough reliable information, clearly state
    that there is insufficient information rather than making up an answer.
11. Preserve relevant source information so that references can be provided.

The response should be educational, neutral, clear, and easy for a patient to understand.
"""

# Quiz Generation Prompt
QUIZ_PROMPT = """
You are HealthBot, an AI-powered patient education assistant.
Your task is to create ONE comprehension question based ONLY on the
patient-friendly summary provided below.

SUMMARY:
{summary}

Instructions:
1. Generate exactly ONE comprehension question.
2. Use ONLY information contained in the summary.
3. Do NOT use outside knowledge.
4. Do NOT introduce facts that are not present in the summary.
5. The question must be answerable using the summary alone.
6. The question should test whether the patient understood an important
   concept from the summary.
7. Do not ask for a diagnosis.
8. Do not ask for personalized medical advice.
9. Do not generate multiple questions.
10. Return only the question.

Question:
"""

# Grade Evaluation Prompt
GRADING_PROMPT = """
You are HealthBot, an AI-powered patient education assistant.
Your task is to evaluate a patient's answer to a comprehension question.
Use ONLY the provided summary as the source of truth.

SUMMARY:
{summary}

QUESTION:
{quiz_question}

PATIENT ANSWER:
{user_answer}

Instructions:
1. Evaluate the patient's answer using ONLY the information in the summary.
2. Do NOT use outside knowledge.
3. Do not introduce medical facts that are not present in the summary.
4. Determine whether the answer demonstrates understanding of the question.
5. Assign one grade:
   - A = Correct and complete
   - B = Mostly correct
   - C = Partially correct
   - D = Incorrect, unsupported, or does not answer the question
6. Explain clearly why the grade was assigned.
7. Identify relevant evidence from the summary.
8. Include the relevant source or citation associated with that evidence when available.
9. Keep the feedback concise and patient-friendly.
10. Do not diagnose the patient or provide personalized medical advice.

Return the result using this structure:
Grade: <A/B/C/D>

Explanation:
<Why the grade was assigned>

Evidence:
<Relevant information from the summary>

Source:
<Relevant source or citation>
"""

TOPIC_CLASSIFICATION_PROMPT = """
You are a health-topic classifier for HealthBot.
Determine whether the following topic is related
to health, medicine, healthcare, disease,
symptoms, nutrition, fitness, physical wellbeing,
mental wellbeing, medical treatment, or prevention.

Topic:
{topic}

Respond with exactly one word:
HEALTH or NON_HEALTH
"""

## Health Topic Vocabulary

In [8]:
# Health Vocabulary
DISEASES_AND_CONDITIONS = {
    "diabetes",
    "type 1 diabetes",
    "type 2 diabetes",
    "prediabetes",
    "hypertension",
    "high blood pressure",
    "low blood pressure",
    "high cholesterol",
    "heart disease",
    "coronary artery disease",
    "heart attack",
    "stroke",
    "asthma",
    "copd",
    "pneumonia",
    "bronchitis",
    "tuberculosis",
    "flu",
    "influenza",
    "covid",
    "covid-19",
    "common cold",
    "cancer",
    "breast cancer",
    "lung cancer",
    "prostate cancer",
    "skin cancer",
    "arthritis",
    "osteoarthritis",
    "rheumatoid arthritis",
    "osteoporosis",
    "migraine",
    "epilepsy",
    "anemia",
    "thyroid disease",
    "hypothyroidism",
    "hyperthyroidism",
    "pcos",
    "endometriosis",
    "ibs",
    "gerd",
    "kidney disease",
    "liver disease",
    "fatty liver",
    "hepatitis",
    "ulcer",
    "gastritis",
    "obesity",
    "depression",
    "anxiety",
    "insomnia",
    "dementia",
    "alzheimer's disease",
    "parkinson's disease"
}


SYMPTOMS = {
    "fever",
    "cough",
    "headache",
    "migraine",
    "fatigue",
    "weakness",
    "dizziness",
    "nausea",
    "vomiting",
    "diarrhea",
    "constipation",
    "abdominal pain",
    "chest pain",
    "back pain",
    "joint pain",
    "muscle pain",
    "sore throat",
    "runny nose",
    "shortness of breath",
    "difficulty breathing",
    "rapid heartbeat",
    "palpitations",
    "swelling",
    "rash",
    "itching",
    "bleeding",
    "weight loss",
    "weight gain",
    "loss of appetite",
    "insomnia"
}


BODY_SYSTEMS = {
    "heart",
    "brain",
    "lungs",
    "kidneys",
    "liver",
    "stomach",
    "intestines",
    "pancreas",
    "thyroid",
    "skin",
    "bones",
    "muscles",
    "blood",
    "immune system",
    "nervous system",
    "digestive system",
    "respiratory system",
    "cardiovascular system"
}


HEALTH_AND_WELLNESS = {
    "nutrition",
    "healthy diet",
    "diet",
    "exercise",
    "physical activity",
    "sleep",
    "sleep health",
    "mental health",
    "stress",
    "stress management",
    "weight management",
    "hydration",
    "vaccination",
    "vaccines",
    "immunization",
    "first aid",
    "hygiene",
    "preventive care",
    "health screening",
    "sexual health",
    "maternal health",
    "child health",
    "elderly health"
}


MEDICAL_CONCEPTS = {
    "medication",
    "medicine",
    "drug",
    "treatment",
    "therapy",
    "side effects",
    "dosage",
    "surgery",
    "physical therapy",
    "chemotherapy",
    "radiation therapy",
    "immunotherapy",
    "diagnosis",
    "prevention",
    "symptoms",
    "causes",
    "risk factors"
}


HEALTH_TOPICS = (
    DISEASES_AND_CONDITIONS
    | SYMPTOMS
    | BODY_SYSTEMS
    | HEALTH_AND_WELLNESS
    | MEDICAL_CONCEPTS
)

print(f"Loaded {len(HEALTH_TOPICS)} health topics.")

Loaded 140 health topics.


## Health Topic Validation

In [9]:
# Normalize a user-provided topic for comparison.
def normalize_topic(topic: str) -> str:
    if not topic:
        return ""
    topic = topic.lower().strip()
    # Replace common separators with spaces
    topic = re.sub(r"[-_/]", " ", topic)
    # Remove extra whitespace
    topic = re.sub(r"\s+", " ", topic)
    return topic

# Exact search in local vocabulary
def exact_health_match(topic: str) -> bool:
    normalized_topic = normalize_topic(topic)
    return normalized_topic in HEALTH_TOPICS

# Fuzzy string matching
FUZZY_THRESHOLD = 85
def fuzzy_health_match(topic: str):
    normalized_topic = normalize_topic(topic)
    if not normalized_topic:
        return None, 0
    result = process.extractOne(
        normalized_topic,
        list(HEALTH_TOPICS),
        scorer=fuzz.ratio
    )
    if result is None:
        return None, 0
    matched_topic, score, _ = result
    if score >= FUZZY_THRESHOLD:
        return matched_topic, score
    return None, score

# Lazy Embedding Model Configuration
embedding_model = None
health_topic_list = sorted(HEALTH_TOPICS)
health_topic_embeddings = None

def load_embedding_model():
    global embedding_model
    global health_topic_embeddings
    if embedding_model is not None:
        return
    print("Loading local biomedical embedding model...")
    embedding_model = SentenceTransformer(
        "NeuML/pubmedbert-base-embeddings"
    )
    health_topic_embeddings = embedding_model.encode(
        health_topic_list,
        normalize_embeddings=True
    )
    health_topic_embeddings = np.asarray(
        health_topic_embeddings
    )
    print("Embedding model loaded successfully.")
    print(
        f"Embedding vocabulary size: "
        f"{len(health_topic_list)}"
    )

# Compare the user topic against the local health vocabulary using cosine similarity. (embedding model)
def get_health_similarity(topic: str):
    normalized_topic = normalize_topic(topic)

    if not normalized_topic:
        return None, 0.0

    # Load model only when needed
    load_embedding_model()

    query_embedding = embedding_model.encode(
        [normalized_topic],
        normalize_embeddings=True
    )
    query_embedding = np.asarray(
        query_embedding
    )
    similarities = (
        query_embedding[0]
        @ health_topic_embeddings.T
    )
    best_index = int(
        np.argmax(similarities)
    )
    best_score = float(
        similarities[best_index]
    )
    best_topic = health_topic_list[
        best_index
    ]

    return best_topic, best_score

# Gemini topic Classifier, used when local vocabulary is inconclusive
def classify_with_gemini(topic: str) -> bool:

    if not topic or not topic.strip():
        return False

    prompt = TOPIC_CLASSIFICATION_PROMPT.format(
        topic=topic
    )

    try:
        response = generate_with_gemini(prompt)
        classification = response.strip().upper()

        if classification == "HEALTH":
            return True
        if classification == "NON_HEALTH":
            return False
        print(
            f"Unexpected Gemini classification: "
            f"{response}"
        )
        return False

    except Exception as e:

        print(
            f"Health classification failed: {e}"
        )
        return False

In [10]:
# Complete validation


EMBEDDING_THRESHOLD = 0.75

def validate_health_topic(topic: str) -> bool:
    """
    Complete health-topic validation pipeline.

    Order:
    1. Exact match
    2. Fuzzy match
    3. Embedding similarity
    4. Gemini fallback
    """

    if not topic or not topic.strip():
        return False
    
    normalized_topic = normalize_topic(
        topic
    )

    # 1. Exact match
    if normalized_topic in HEALTH_TOPICS:
        print(
            "Validation method: Exact match"
        )
        return True

    # 2. Fuzzy match
    fuzzy_match, fuzzy_score = (
        fuzzy_health_match(
            normalized_topic
        )
    )
    if fuzzy_match is not None:
        print(
            f"Validation method: Fuzzy match"
        )
        print(
            f"Matched topic: {fuzzy_match}"
        )
        print(
            f"Fuzzy score: {fuzzy_score:.1f}"
        )
        return True

    # 3. Embedding similarity
    embedding_match, embedding_score = (
        get_health_similarity(
            normalized_topic
        )
    )
    print(
        f"Semantic match: {embedding_match}"
    )
    print(
        f"Semantic score: {embedding_score:.3f}"
    )
    if (
        embedding_score
        >= EMBEDDING_THRESHOLD
    ):
        print(
            "Validation method: Embedding"
        )
        return True

    # 4. Gemini fallback
    print(
        "Using Gemini classifier..."
    )
    return classify_with_gemini(
        normalized_topic
    )


In [ ]:
# Test health topic validation

test_topics = [
    "diabetes",
    "pressure problem",
    "trouble sleeping",
    "stock market"
]

for topic in test_topics:

    print("\n" + "=" * 60)
    print(f"Topic: {topic}")

    result = validate_health_topic(topic)

    print(f"Final decision: {result}")

## LangGraph Workflow Nodes

This section implements the individual LangGraph nodes that perform the HealthBot workflow.

Each node has a single responsibility and communicates with other nodes through the shared `HealthBotState`.

### - Get Topic Node

In [11]:
def get_topic(state: HealthBotState) -> HealthBotState:

    while True:

        topic = input(
            "\nWhat health topic or medical condition "
            "would you like to learn about?\n> "
        ).strip()

        # Empty input
        if not topic:

            print(
                "Please enter a health topic."
            )

            continue

        # Health topic validation
        if not validate_health_topic(topic):

            print(
                "\nThe topic does not appear to be "
                "health-related."
            )

            print(
                "Please enter a medical or health topic."
            )

            continue

        # Valid topic
        print(
            f"\nHealth topic accepted: {topic}"
        )

        return {
            "topic": topic,
            "error": ""
        }

In [ ]:
# Test get_topic Node

test_state: HealthBotState = {}

topic_result = get_topic(test_state)

print("=" * 60)
print("GET TOPIC TEST")
print("=" * 60)

if topic_result.get("error"):
    print("Error:", topic_result["error"])
else:
    print("Topic:", topic_result["topic"])

print("=" * 60)

In [ ]:
# Test invalid topic

invalid_topic_state: HealthBotState = {}

result = get_topic(invalid_topic_state)

print(result)

### - Search Node
Search for medical information based on the validated topic.

In [12]:
def search_node(state: HealthBotState) -> HealthBotState:
    
    topic = state.get("topic", "")

    if not is_valid_text(topic):
        return {
            "search_results": [],
            "error": "Health topic is missing."
        }

    try:
        results = search_medical_information(topic)

        if not results:
            return {
                "search_results": [],
                "error": "No medical information found."
            }

        return {
            "search_results": results,
            "error": ""
        }

    except Exception as e:
        return {
            "search_results": [],
            "error": f"Medical search failed: {e}"
        }

In [ ]:
# Test search_node

search_test_state: HealthBotState = {
    "topic": topic_result.get("topic", "")
}

search_result = search_node(search_test_state)

print("=" * 60)
print("SEARCH NODE TEST")
print("=" * 60)

if search_result.get("error"):
    print("Error:", search_result["error"])
else:
    search_results = search_result.get("search_results", [])

    print(f"Results returned: {len(search_results)}")

    for i, result in enumerate(search_results, start=1):
        print(f"\n--- Result {i} ---")
        print("Title:", result.get("title", ""))
        print("URL:", result.get("url", ""))

print("=" * 60)

In [ ]:
# Test search_node with invalid input

invalid_search_state: HealthBotState = {
    "topic": ""
}

result = search_node(invalid_search_state)

print(result)

### - Summarization Node
Generate a patient-friendly summary from Tavily results.

In [13]:
def summarize_information(state: HealthBotState) -> HealthBotState:
    topic = state.get("topic", "")
    search_results = state.get("search_results", [])

    if not is_valid_text(topic):
        return {
            "summary": "",
            "error": "Health topic is missing."
        }

    if not isinstance(search_results, list) or not search_results:
        return {
            "summary": "",
            "error": "No medical information available for summarization."
        }

    formatted_results = []

    for result in search_results:

        if not isinstance(result, dict):
            continue

        content = result.get("content", "")

        if not is_valid_text(content):
            continue

        formatted_results.append(
            f"Title: {result.get('title', 'N/A')}\n"
            f"URL: {result.get('url', 'N/A')}\n"
            f"Content: {content}"
        )

    if not formatted_results:
        return {
            "summary": "",
            "error": "Search results contain no usable information."
        }

    search_context = "\n\n".join(formatted_results)

    prompt = format_prompt(
        SUMMARY_PROMPT,
        topic=topic,
        search_results=search_context
    )

    try:
        summary = generate_with_gemini(prompt)

        if not is_valid_text(summary):
            return {
                "summary": "",
                "error": "Gemini returned an empty summary."
            }

        return {
            "summary": summary.strip(),
            "error": ""
        }

    except Exception as e:
        return {
            "summary": "",
            "error": f"Summarization failed: {e}"
        }

In [ ]:
# Testing Summarization Node

summary_test_state: HealthBotState = {
    "topic": topic_result.get("topic", ""),
    "search_results":  search_result["search_results"]
}

summary_result = summarize_information(summary_test_state)

print("=" * 60)
print("SUMMARY TEST")
print("=" * 60)

if summary_result.get("error"):
    print("Error:", summary_result["error"])
else:
    print(summary_result["summary"])

print("=" * 60)

In [ ]:
# Test summarize_information with no search results

invalid_summary_state: HealthBotState = {
    "topic": "diabetes",
    "search_results": []
}

result = summarize_information(invalid_summary_state)

print(result)

### - Display Summary Node

In [14]:
def display_summary(state: HealthBotState) -> HealthBotState:
    summary = state.get("summary", "").strip()

    if not summary:
        return {
            "ready_for_quiz": False,
            "error": "Summary is unavailable."
        }

    print("\n" + "=" * 60)
    print("HEALTH INFORMATION")
    print("=" * 60)
    print(summary)
    print("=" * 60)

    while True:
        response = input(
            "\nAre you ready for the comprehension check? (yes/no)\n> "
        ).strip().lower()

        if response in {"yes", "y"}:
            return {
                "ready_for_quiz": True,
                "error": ""
            }

        if response in {"no", "n"}:
            return {
                "ready_for_quiz": False,
                "error": ""
            }

        print("Invalid input. Please enter yes or no.")

In [ ]:
# Test display_summary Node

display_test_state: HealthBotState = {
    "topic": topic_result.get("topic", ""),
    "summary": summary_result.get("summary", "")
}

display_result = display_summary(display_test_state)

print("\nResult:")
print(display_result)

### - Quiz Generation Node
Generate one comprehension question based only on the summary.

In [15]:
def generate_quiz(state: HealthBotState) -> HealthBotState:
    
    summary = state.get("summary", "")

    if not is_valid_text(summary):
        return {
            "quiz_question": "",
            "ready_for_quiz": False,
            "error": "Summary is missing. Cannot generate quiz."
        }

    prompt = format_prompt(
        QUIZ_PROMPT,
        summary=summary
    )

    try:
        question = generate_with_gemini(prompt)

        if not is_valid_text(question):
            return {
                "quiz_question": "",
                "ready_for_quiz": False,
                "error": "Gemini returned an empty quiz question."
            }

        return {
            "quiz_question": question.strip(),
            "ready_for_quiz": True,
            "error": ""
        }

    except Exception as e:
        return {
            "quiz_question": "",
            "ready_for_quiz": False,
            "error": f"Quiz generation failed: {e}"
        }

In [ ]:
# Test generate_quiz Node

quiz_test_state: HealthBotState = {
    "summary": summary_result.get("summary", "")
}

quiz_result = generate_quiz(quiz_test_state)

print("=" * 60)
print("QUIZ GENERATION NODE TEST")
print("=" * 60)

if quiz_result.get("error"):
    print("Error:", quiz_result["error"])
else:
    print("Question:")
    print(quiz_result["quiz_question"])

print("=" * 60)

In [ ]:
# Test generate_quiz with empty summary

invalid_quiz_state: HealthBotState = {
    "summary": ""
}

result = generate_quiz(invalid_quiz_state)

print(result)

### - Get Quiz Answer Node
Collect the user's answer to the generated quiz question.

In [16]:
def get_quiz_answer(state: HealthBotState) -> HealthBotState:
    
    question = state.get("quiz_question", "")

    if not is_valid_text(question):
        return {
            "user_answer": "",
            "error": "Quiz question is missing."
        }

    while True:
        answer = input(f"\nQuestion:\n{question}\n\nYour answer:\n> ").strip()
        if not answer:
            print("Please provide an answer before continuing.")
            continue

        return {
            "user_answer": answer,
            "error": ""
        }

In [ ]:
# Test get_quiz_answer Node

answer_test_state: HealthBotState = {
    "quiz_question": quiz_result.get("quiz_question", "")
}

answer_result = get_quiz_answer(answer_test_state)

print("\nResult:")
print(answer_result)

### - Grade Answer Node
Grade the user's answer using only the generated summary.

In [17]:
def grade_answer(state: HealthBotState) -> HealthBotState:
    
    topic = state.get("topic", "")
    summary = state.get("summary", "")
    quiz_question = state.get("quiz_question", "")
    user_answer = state.get("user_answer", "")

    if not is_valid_text(topic):
        return {
            "grade": "",
            "feedback": "",
            "error": "Health topic is missing."
        }

    if not is_valid_text(summary):
        return {
            "grade": "",
            "feedback": "",
            "error": "Summary is missing."
        }

    if not is_valid_text(quiz_question):
        return {
            "grade": "",
            "feedback": "",
            "error": "Quiz question is missing."
        }

    if not is_valid_text(user_answer):
        return {
            "grade": "",
            "feedback": "",
            "error": "User answer is missing."
        }

    prompt = format_prompt(
        GRADING_PROMPT,
        topic=topic,
        summary=summary,
        quiz_question=quiz_question,
        user_answer=user_answer
    )

    try:

        grading_result = generate_with_gemini(prompt)

        if not is_valid_text(grading_result):
            return {
            "grade": "",
            "feedback": "",
            "error": "Gemini returned an empty grading response."
            }

        grade = extract_grade(grading_result)

        if not grade:
            return {
            "grade": "",
            "feedback": grading_result.strip(),
            "error": "Gemini returned an invalid grade format."
            }

        return {
        "grade": grade,
        "feedback": grading_result.strip(),
        "error": ""
        }

    except Exception as e:

        return {
        "grade": "",
        "feedback": "",
        "error": f"Answer grading failed: {e}"
        }

In [ ]:
# Test grade_answer Node

grading_test_state: HealthBotState = {
    "topic": topic_result.get("topic", ""),
    "summary": summary_result.get("summary", ""),
    "quiz_question": quiz_result.get("quiz_question", ""),
    "user_answer": answer_result.get("user_answer", "")
}

grading_result = grade_answer(grading_test_state)

print("=" * 60)
print("ANSWER GRADING NODE TEST")
print("=" * 60)

if grading_result.get("error"):
    print("Error:", grading_result["error"])
else:
    print("Grade:", grading_result.get("grade", "N/A"))
    print("\nFeedback:")
    print(grading_result.get("feedback", "N/A"))

print("=" * 60)

In [ ]:
# Test grade_answer with empty answer

invalid_grading_state: HealthBotState = {
    "topic": "Diabetes",
    "summary": "Diabetes affects how the body manages blood glucose.",
    "quiz_question": "What does diabetes affect?",
    "user_answer": ""
}

result = grade_answer(invalid_grading_state)

print(result)

### - Display Feedback Node
Display the grading result and feedback to the user.

In [18]:
def display_feedback(state: HealthBotState) -> HealthBotState:
    
    grade = state.get("grade", "")
    feedback = state.get("feedback", "")
    error = state.get("error", "")

    if error:
        print(f"\nUnable to evaluate your answer: {error}")
        return state

    if not is_valid_text(feedback):
        print("\nNo feedback was generated.")
        return {
            "error": "Feedback is unavailable."
        }

    print("\n" + "=" * 50)
    print("QUIZ RESULT")
    print("=" * 50)

    if grade:
        print(f"\nGrade: {grade}")

    print("\nFeedback:")
    print(feedback)

    print("=" * 50)

    return state

In [ ]:
# Test display_feedback Node

feedback_test_state: HealthBotState = {
    "feedback": grading_result.get("feedback", "")
}

feedback_result = display_feedback(feedback_test_state)

print("\nResult:")
print(feedback_result)

### - Session Decision Node
Ask whether the user wants to learn about another topic.

In [19]:
def session_decision(state: HealthBotState) -> HealthBotState:
    
    while True:
        choice = input(
            "\nWould you like to learn about another health topic? "
            "(yes/no)\n> "
        ).strip().lower()

        if choice in {"yes", "y"}:
            return {
                "continue_session": True,
                "error": ""
            }

        if choice in {"no", "n"}:
            return {
                "continue_session": False,
                "error": ""
            }

        print(
            "Please enter 'yes' or 'no'."
        )

In [ ]:
# Test session_decision Node

session_test_state: HealthBotState = {}

session_result = session_decision(session_test_state)

print("\nResult:")
print(session_result)

### - Reset State Node
Reset session-specific data for a new health topic.

In [20]:
def reset_state_node(state: HealthBotState) -> HealthBotState:
    return reset_state()

In [ ]:
# Test reset_state_node

reset_test_state: HealthBotState = {
    "topic": "diabetes",
    "search_results": [{"title": "Test Source"}],
    "summary": "Test summary",
    "quiz_question": "Test question",
    "user_answer": "Test answer",
    "grade": "A",
    "feedback": "Test feedback",
    "ready_for_quiz": True,
    "continue_session": True,
    "error": ""
}

reset_result = reset_state_node(reset_test_state)

print("=" * 60)
print("RESET STATE NODE TEST")
print("=" * 60)

print(reset_result)

print("=" * 60)

## Langraph Workflow Construction

In [21]:
# Create the HealthBot state graph

healthbot_graph = StateGraph(HealthBotState)

print("HealthBot state graph created.")

HealthBot state graph created.


In [22]:
# Add workflow nodes to the graph

healthbot_graph.add_node("get_topic", get_topic)
healthbot_graph.add_node("search", search_node)
healthbot_graph.add_node("summarize", summarize_information)
healthbot_graph.add_node("display_summary", display_summary)
healthbot_graph.add_node("generate_quiz", generate_quiz)
healthbot_graph.add_node("get_quiz_answer", get_quiz_answer)
healthbot_graph.add_node("grade_answer", grade_answer)
healthbot_graph.add_node("display_feedback", display_feedback)
healthbot_graph.add_node("session_decision", session_decision)
healthbot_graph.add_node("reset_state", reset_state_node)

print("All HealthBot workflow nodes added.")

All HealthBot workflow nodes added.


In [23]:
# Connect the main workflow nodes

healthbot_graph.add_edge(
    START,
    "get_topic"
)

healthbot_graph.add_edge(
    "get_topic",
    "search"
)

healthbot_graph.add_edge(
    "search",
    "summarize"
)

healthbot_graph.add_edge(
    "summarize",
    "display_summary"
)

healthbot_graph.add_edge(
    "display_summary",
    "generate_quiz"
)

healthbot_graph.add_edge(
    "generate_quiz",
    "get_quiz_answer"
)

healthbot_graph.add_edge(
    "get_quiz_answer",
    "grade_answer"
)

healthbot_graph.add_edge(
    "grade_answer",
    "display_feedback"
)

healthbot_graph.add_edge(
    "display_feedback",
    "session_decision"
)

print("Main workflow edges added.")

Main workflow edges added.


In [24]:
# Decide whether to start another health-topic session or end the workflow.
def route_session(state: HealthBotState) -> str:

    if state.get("continue_session", False):
        return "reset_state"

    return END

In [25]:
# Add conditional routing after session decision

healthbot_graph.add_conditional_edges(
    "session_decision",
    route_session,
    {
        "reset_state": "reset_state",
        END: END
    }
)

print("Conditional session routing added.")

Conditional session routing added.


In [26]:
# Start a new topic after resetting the state

healthbot_graph.add_edge(
    "reset_state",
    "get_topic"
)

print("Session reset loop added.")

Session reset loop added.


In [27]:
# Compile the HealthBot workflow

healthbot_app = healthbot_graph.compile()

print("HealthBot workflow compiled successfully.")

HealthBot workflow compiled successfully.


In [28]:
# Display the workflow structure

try:
    display(
        healthbot_app.get_graph().draw_mermaid()
    )
except Exception as e:
    print(f"Unable to display graph visualization: {e}")

'---\nconfig:\n  flowchart:\n    curve: linear\n---\ngraph TD;\n\t__start__([<p>__start__</p>]):::first\n\tget_topic(get_topic)\n\tsearch(search)\n\tsummarize(summarize)\n\tdisplay_summary(display_summary)\n\tgenerate_quiz(generate_quiz)\n\tget_quiz_answer(get_quiz_answer)\n\tgrade_answer(grade_answer)\n\tdisplay_feedback(display_feedback)\n\tsession_decision(session_decision)\n\treset_state(reset_state)\n\t__end__([<p>__end__</p>]):::last\n\t__start__ --> get_topic;\n\tdisplay_feedback --> session_decision;\n\tdisplay_summary --> generate_quiz;\n\tgenerate_quiz --> get_quiz_answer;\n\tget_quiz_answer --> grade_answer;\n\tget_topic --> search;\n\tgrade_answer --> display_feedback;\n\treset_state --> get_topic;\n\tsearch --> summarize;\n\tsession_decision -.-> __end__;\n\tsession_decision -.-> reset_state;\n\tsummarize --> display_summary;\n\tclassDef default fill:#f2f0ff,line-height:1.2\n\tclassDef first fill-opacity:0\n\tclassDef last fill:#bfb6fc\n'

In [29]:
# Verify registered workflow nodes

graph_nodes = healthbot_app.get_graph().nodes

print("Registered workflow nodes:")

for node_name in graph_nodes:
    print("-", node_name)

Registered workflow nodes:
- __start__
- get_topic
- search
- summarize
- display_summary
- generate_quiz
- get_quiz_answer
- grade_answer
- display_feedback
- session_decision
- reset_state
- __end__


### Run the Complete Application

In [32]:
# Run HealthBot

initial_state = reset_state()

final_state = healthbot_app.invoke(
    initial_state
)

print("\nHealthBot session completed.")

Loading local biomedical embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Embedding model loaded successfully.
Embedding vocabulary size: 140
Semantic match: runny nose
Semantic score: 0.476
Using Gemini classifier...

Health topic accepted: broken leg

HEALTH INFORMATION
A broken leg, which healthcare providers also call a leg fracture, is a break or crack in one of the three bones in your leg. Common causes of a broken leg include falls, motor vehicle accidents, and sports injuries, such as extending your leg beyond its natural limits or taking a direct blow from an opponent or a sports tool like a hockey stick. Additionally, repetitive activities like running can cause "stress fractures," which are small cracks or breaks in the bone that develop over a few weeks. In children, especially those who cannot yet walk, a broken leg can sometimes be a result of child abuse, so healthcare evaluations for children often include routine questions to rule out intentional injury.

The symptoms of a broken leg are often apparent immediately and typically involve excru

## Testing

In [34]:
# Test initial/reset state

test_state = reset_state()

assert test_state["topic"] == ""
assert test_state["search_results"] == []
assert test_state["summary"] == ""
assert test_state["quiz_question"] == ""
assert test_state["user_answer"] == ""
assert test_state["grade"] == ""
assert test_state["feedback"] == ""
assert test_state["ready_for_quiz"] is False
assert test_state["continue_session"] is False
assert test_state["error"] == ""

print("State reset test passed.")

State reset test passed.


In [35]:
# Test health-topic validation

health_topic_tests = {
    "diabetes": True,
    "high blood pressure": True,
    "chest pain": True,
    "diabetees": True,
    "football": False,
    "stock market": False,
    "car engine": False
}

for topic, expected in health_topic_tests.items():

    result = validate_health_topic(topic)

    print(
        f"{topic:25} "
        f"Expected: {expected} "
        f"Actual: {result}"
    )

Validation method: Exact match
diabetes                  Expected: True Actual: True
Validation method: Exact match
high blood pressure       Expected: True Actual: True
Validation method: Exact match
chest pain                Expected: True Actual: True
Validation method: Fuzzy match
Matched topic: diabetes
Fuzzy score: 94.1
diabetees                 Expected: True Actual: True
Semantic match: flu
Semantic score: 0.417
Using Gemini classifier...
football                  Expected: False Actual: False
Semantic match: medication
Semantic score: 0.320
Using Gemini classifier...
stock market              Expected: False Actual: False
Semantic match: fatigue
Semantic score: 0.318
Using Gemini classifier...
car engine                Expected: False Actual: False


In [36]:
# Test search node with missing topic

search_error_state: HealthBotState = {
    "topic": "",
    "search_results": []
}

search_error_result = search_node(
    search_error_state
)

assert search_error_result.get("error") == (
    "Health topic is missing."
)

print("Search node invalid-input test passed.")

Search node invalid-input test passed.


In [37]:
# Test summarization with no search results

summary_error_state: HealthBotState = {
    "topic": "diabetes",
    "search_results": []
}

summary_error_result = summarize_information(
    summary_error_state
)

assert summary_error_result.get("error") == (
    "No medical information available for summarization."
)

print("Summarization invalid-input test passed.")

Summarization invalid-input test passed.


In [38]:
# Test quiz generation with missing summary

quiz_error_state: HealthBotState = {
    "summary": ""
}

quiz_error_result = generate_quiz(
    quiz_error_state
)

assert quiz_error_result.get("ready_for_quiz") is False

assert quiz_error_result.get("error") == (
    "Summary is missing. Cannot generate quiz."
)

print("Quiz invalid-input test passed.")

Quiz invalid-input test passed.


In [39]:
# Test grading with missing answer

grading_error_state: HealthBotState = {
    "topic": "diabetes",
    "summary": (
        "Diabetes is a condition that affects "
        "blood sugar regulation."
    ),
    "quiz_question": (
        "What does diabetes affect?"
    ),
    "user_answer": ""
}

grading_error_result = grade_answer(
    grading_error_state
)

assert grading_error_result.get("error") == (
    "User answer is missing."
)

print("Grading invalid-input test passed.")

Grading invalid-input test passed.


In [40]:
# Test grade extraction

assert extract_grade("Grade: A") == "A"
assert extract_grade("Grade: B\nExplanation: Mostly correct.") == "B"
assert extract_grade("Grade: C") == "C"
assert extract_grade("Grade: D") == "D"
assert extract_grade("Invalid response") == ""

print("Grade extraction tests passed.")

Grade extraction tests passed.


In [41]:
# Test feedback display

feedback_test_state: HealthBotState = {
    "grade": "A",
    "feedback": (
        "Grade: A\n"
        "Explanation: Correct and complete.\n"
        "Evidence: The answer matches the summary."
    ),
    "error": ""
}

feedback_result = display_feedback(
    feedback_test_state
)

assert feedback_result["grade"] == "A"
assert feedback_result["error"] == ""

print("Display feedback test passed.")


QUIZ RESULT

Grade: A

Feedback:
Grade: A
Explanation: Correct and complete.
Evidence: The answer matches the summary.
Display feedback test passed.


In [42]:
# Test session routing

continue_state: HealthBotState = {
    "continue_session": True
}

end_state: HealthBotState = {
    "continue_session": False
}

assert route_session(continue_state) == "reset_state"
assert route_session(end_state) == END

print("Session routing tests passed.")

Session routing tests passed.


In [43]:
# Test reset state node

reset_test_state: HealthBotState = {
    "topic": "diabetes",
    "summary": "Some summary",
    "quiz_question": "What is diabetes?",
    "user_answer": "A condition",
    "grade": "A",
    "feedback": "Correct",
    "continue_session": True,
    "error": ""
}

reset_result = reset_state_node(
    reset_test_state
)

assert reset_result["topic"] == ""
assert reset_result["summary"] == ""
assert reset_result["quiz_question"] == ""
assert reset_result["user_answer"] == ""
assert reset_result["grade"] == ""
assert reset_result["feedback"] == ""
assert reset_result["continue_session"] is False
assert reset_result["error"] == ""

print("Reset node test passed.")

Reset node test passed.
